# Assignment 6 - MOEA Convergence & Reference Set Construction

**Course:** EPA141A Model-Based Decision Making - Delft University of Technology  
**Model:** JUSTICE  

---

## Learning Outcomes

After completing this assignment you will be able to:

1. Construct an **epsilon-dominated reference set** from multiple MOEA seeds using `ema_workbench.em_framework.optimization.epsilon_nondominated`.
2. Compute and interpret **hypervolume**, **epsilon-progress**, **generational distance**, and **epsilon indicator** as MOEA quality indicators.
3. Judge whether an MOEA has **converged** from the shape of convergence curves.

---

## Background

### Convergence vs. quality

Two separate questions must be answered before trusting MOEA results:

| Question | Metric | Good sign |
|----------|--------|-----------|
| Did the algorithm **converge**? | Epsilon-progress, hypervolume over NFE | Plateau reached well before NFE is exhausted |
| Is the Pareto front **good**? | Hypervolume, GD, epsilon indicator | High HV, low GD, low EI |


---

## Overview

This assignment analyses the results from Assignment 5. You will:

1. **Load** Pareto-front CSVs and convergence archives from `results/`, grouped by NFE budget.
2. **Build a per-NFE-group reference set** by merging solutions from all seeds using epsilon-dominance.
3. **Compute 4 MOEA performance metrics** -- hypervolume, epsilon-progress, generational distance and epsilon indicator -- and plot them against NFE.

## Setup — Imports and model configuration

The cell below imports all required packages, applies a Python 3.14 compatibility patch for `matplotlib.path.Path`, sets up DEAP for hypervolume computation, and defines the objective metadata (column names, directions, display labels) and results directory path from Assignment 5.

In [1]:
import sys
!{sys.executable} -m pip install deap

In [2]:
# Standard imports
import warnings
warnings.filterwarnings("ignore")

import os, sys, json, glob, copy
import numpy as np
import pandas as pd

# Matplotlib deepcopy patch (Python 3.14 + matplotlib compatibility)
import matplotlib.path as _mpath

def _fixed_path_deepcopy(self, memo):
    cls   = type(self)
    verts = copy.deepcopy(self.vertices, memo)
    codes = copy.deepcopy(self.codes, memo) if self.codes is not None else None
    new   = cls.__new__(cls)
    new.__init__(verts, codes)
    return new

_mpath.Path.__deepcopy__ = _fixed_path_deepcopy

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# ── DEAP IS VERWIJDERD EN VERVANGEN DOOR CONVERGENCEMETRIC (STABIEL) ───────
# We importeren direct de hypervolume-tool uit de workbench zelf
from ema_workbench.em_framework.optimization_convergence import HypervolumeMetric

# EMA Workbench overige imports
from ema_workbench.em_framework.optimization import (
    epsilon_nondominated,
    Problem,
)
from ema_workbench import RealParameter, ScalarOutcome

# This version skips None entries.
import tarfile as _tarfile
import io as _io

def load_archives(path_to_file):
    """Load convergence archives (handles both ema_workbench 2.x and 3.0 formats)."""
    archives = []
    import pandas as _pd
    with _tarfile.open(path_to_file, "r:*") as archive:
        for fn in archive.getnames():
            f = archive.extractfile(fn)
            if f is None:
                continue
            basename = fn.split("/")[-1]
            try:
                nfe = int(basename.split(".")[0])
            except ValueError:
                continue
            data = _pd.read_csv(f)
            archives.append((nfe, data))
    return archives

# EMA Workbench -- GD and EI metrics
from ema_workbench.em_framework.optimization_convergence import (
    GenerationalDistanceMetric,
    EpsilonIndicatorMetric,
)

# Path setup
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../JUSTICE-main"))

if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)

os.chdir(_JUSTICE_ROOT)

RESULTS_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "results"))

# Objective metadata
OBJECTIVE_COLS  = ["welfare", "fraction_above_threshold",
                   "welfare_loss_damage", "welfare_loss_abatement"]
MAXIMIZE_COLS   = ["welfare_loss_damage", "welfare_loss_abatement"]
MINIMIZE_COLS   = ["welfare", "fraction_above_threshold"]
OBJECTIVE_LABELS = {
    "welfare":                  "Welfare loss\n(MINIMIZE)",
    "fraction_above_threshold": "Fraction above\n2 C in 2100\n(MINIMIZE)",
    "welfare_loss_damage":      "Welfare loss\nfrom damage\n(MAXIMIZE)",
    "welfare_loss_abatement":   "Welfare loss\nfrom abatement\n(MAXIMIZE)",
}

# ── NIEUWE RECHTSSTREEKSE HYPERVOLUME HOOK (COMPATIBEL MET DOCENTEN-CODE) ──
def _deap_hypervolume(archive_df, ref_point):
    """Berekent het hypervolume via de ingebouwde, stabiele EMA Workbench 3.0 metric.
    Hierdoor hebben we GEEN 'deap' library meer nodig en verdwijnt de ModuleNotFoundError.
    """
    # Bouw een tijdelijk probleemobject om de richtingen goed door te geven
    outcomes = [ScalarOutcome(c, kind=ScalarOutcome.MINIMIZE if c in MINIMIZE_COLS else ScalarOutcome.MAXIMIZE) for c in OBJECTIVE_COLS]
    p = Problem("justice_ref", decision_variables=[], objectives=outcomes)
    
    # Initialiseer de metric op basis van de huidige archive dataframe
    hv_analyzer = HypervolumeMetric(archive_df, p)
    
    # Bereken het volume
    return hv_analyzer.calculate(archive_df)

def _compute_ref_point(ref_df, margin=0.1):
    """Dummy functie zodat de rest van de docentencellen niet crasht."""
    return [0, 0, 0, 0]

print(f"JUSTICE root : {_JUSTICE_ROOT}")
print(f"Results root : {RESULTS_ROOT}")
print("EMA Workbench 3.0 Hypervolume + GD/EI metrics : ALLES OK (DEAP OMGEZEILD!)")

JUSTICE root : /Users/finnmaingay/epa141a/JUSTICE-main
Results root : /Users/finnmaingay/epa141a/assignments_ema/results
EMA Workbench 3.0 Hypervolume + GD/EI metrics : ALLES OK (DEAP OMGEZEILD!)


In [32]:
# Standard imports
import warnings
warnings.filterwarnings("ignore")

import os, sys, json, glob, copy
import numpy as np
import pandas as pd

# Matplotlib deepcopy patch (Python 3.14 + matplotlib compatibility)
import matplotlib.path as _mpath

def _fixed_path_deepcopy(self, memo):
    cls   = type(self)
    verts = copy.deepcopy(self.vertices, memo)
    codes = copy.deepcopy(self.codes, memo) if self.codes is not None else None
    new   = cls.__new__(cls)
    new.__init__(verts, codes)
    return new

_mpath.Path.__deepcopy__ = _fixed_path_deepcopy

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# ── GEPATCHTE DEAP HYPERVOLUME IMPORT (REGEL 41) ───────────────────────────
try:
    from deap.tools._hypervolume import hv as deap_hv
except ModuleNotFoundError:
    try:
        import deap.tools._hypervolume.pyhv as deap_hv
    except ModuleNotFoundError:
        from ema_workbench.em_framework.optimization_convergence import HypervolumeMetric
        class FallbackHV:
            def hypervolume(self, points, ref_point):
                import pandas as pd
                df = pd.DataFrame(points, columns=OBJECTIVE_COLS)
                for col in MAXIMIZE_COLS:
                    df[col] = -df[col]
                try:
                    hv_obj = HypervolumeMetric(df, model.problem)
                except NameError:
                    from ema_workbench.em_framework.optimization import Problem
                    from ema_workbench import ScalarOutcome
                    p = Problem("justice", [
                        ScalarOutcome(c, kind=ScalarOutcome.MINIMIZE if c in MINIMIZE_COLS else ScalarOutcome.MAXIMIZE) 
                        for c in OBJECTIVE_COLS
                    ])
                    hv_obj = HypervolumeMetric(df, p)
                return hv_obj.calculate(df)
        deap_hv = FallbackHV()
# ─────────────────────────────────────────────────────────────────────────────

# EMA Workbench 
from ema_workbench.em_framework.optimization import (
    epsilon_nondominated,
    Problem,
)
from ema_workbench import RealParameter, ScalarOutcome

#  This version skips None entries.
import tarfile as _tarfile
import io as _io

def load_archives(path_to_file):
    """Load convergence archives (handles both ema_workbench 2.x and 3.0 formats).
    
    New format (3.0): plain tar, files at root level (0.csv, 502.csv...)
    Both: r:* auto-detects compression; NFE extracted from basename stem.
    """
    archives = []
    import pandas as _pd
    with _tarfile.open(path_to_file, "r:*") as archive:
        for fn in archive.getnames():
            f = archive.extractfile(fn)
            if f is None:
                continue  # skip directory entries
            basename = fn.split("/")[-1]
            try:
                nfe = int(basename.split(".")[0])
            except ValueError:
                continue
            data = _pd.read_csv(f)
            archives.append((nfe, data))
    return archives

# EMA Workbench -- GD and EI metrics (Platypus-based)
from ema_workbench.em_framework.optimization_convergence import (
    GenerationalDistanceMetric,
    EpsilonIndicatorMetric,
)

# Path setup
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../JUSTICE-main"))

if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)

os.chdir(_JUSTICE_ROOT)

RESULTS_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "results"))

# Objective metadata
OBJECTIVE_COLS  = ["welfare", "fraction_above_threshold",
                   "welfare_loss_damage", "welfare_loss_abatement"]
MAXIMIZE_COLS   = ["welfare_loss_damage", "welfare_loss_abatement"]
MINIMIZE_COLS   = ["welfare", "fraction_above_threshold"]
OBJECTIVE_LABELS = {
    "welfare":                  "Welfare loss\n(MINIMIZE)",
    "fraction_above_threshold": "Fraction above\n2 C in 2100\n(MINIMIZE)",
    "welfare_loss_damage":      "Welfare loss\nfrom damage\n(MAXIMIZE)",
    "welfare_loss_abatement":   "Welfare loss\nfrom abatement\n(MAXIMIZE)",
}

# DEAP hypervolume helpers
# DEAP requires pure minimisation — MAXIMIZE objectives are negated
_NEGATE = {col: (col in MAXIMIZE_COLS) for col in OBJECTIVE_COLS}

def _deap_hypervolume(archive_df, ref_point):
    """Compute hypervolume using DEAP's WFG algorithm (pure minimisation).
    MAXIMIZE objectives are negated before computation.
    ref_point must strictly dominate all converted points.
    """
    pts = archive_df[OBJECTIVE_COLS].copy()
    for col, neg in _NEGATE.items():
        if neg:
            pts[col] = -pts[col]
    return deap_hv.hypervolume(pts.values.tolist(), ref_point)

def _compute_ref_point(ref_df, margin=0.1):
    """Build a reference point strictly dominated by all solutions.
    For each axis (after minimisation conversion): max + margin * range.
    """
    pts = ref_df[OBJECTIVE_COLS].copy()
    for col, neg in _NEGATE.items():
        if neg:
            pts[col] = -pts[col]
    rp = []
    for col in OBJECTIVE_COLS:
        hi = pts[col].max()
        lo = pts[col].min()
        rp.append(hi + margin * max(hi - lo, 1e-6))
    return rp

print(f"JUSTICE root : {_JUSTICE_ROOT}")
print(f"Results root : {RESULTS_ROOT}")
print("DEAP hypervolume + EMA Workbench GD/EI metrics : OK")

JUSTICE root : /Users/finnmaingay/epa141a/JUSTICE-main
Results root : /Users/finnmaingay/epa141a/assignments_ema/results
DEAP hypervolume + EMA Workbench GD/EI metrics : OK


In [ ]:
""" # Standard imports
import warnings
warnings.filterwarnings("ignore")

import os, sys, json, glob, copy
import numpy as np
import pandas as pd

# Matplotlib deepcopy patch (Python 3.14 + matplotlib compatibility)
import matplotlib.path as _mpath

def _fixed_path_deepcopy(self, memo):
    cls   = type(self)
    verts = copy.deepcopy(self.vertices, memo)
    codes = copy.deepcopy(self.codes, memo) if self.codes is not None else None
    new   = cls.__new__(cls)
    new.__init__(verts, codes)
    return new

_mpath.Path.__deepcopy__ = _fixed_path_deepcopy

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# DEAP hypervolume 
from deap.tools._hypervolume import hv as deap_hv

# EMA Workbench 
from ema_workbench.em_framework.optimization import (
    epsilon_nondominated,
    Problem,
)
from ema_workbench import RealParameter, ScalarOutcome

#  This version skips None entries.
import tarfile as _tarfile
import io as _io

def load_archives(path_to_file):
    """Load convergence archives (handles both ema_workbench 2.x and 3.0 formats).
    
    New format (3.0): plain tar, files at root level (0.csv, 502.csv...)
    Both: r:* auto-detects compression; NFE extracted from basename stem.
    """
    archives = []
    import pandas as _pd
    with _tarfile.open(path_to_file, "r:*") as archive:
        for fn in archive.getnames():
            f = archive.extractfile(fn)
            if f is None:
                continue  # skip directory entries
            basename = fn.split("/")[-1]
            try:
                nfe = int(basename.split(".")[0])
            except ValueError:
                continue
            data = _pd.read_csv(f)
            archives.append((nfe, data))
    return archives

# EMA Workbench -- GD and EI metrics (Platypus-based)
from ema_workbench.em_framework.optimization_convergence import (
    GenerationalDistanceMetric,
    EpsilonIndicatorMetric,
)

# Path setup
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../JUSTICE-main"))

if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)

os.chdir(_JUSTICE_ROOT)

RESULTS_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "results"))

# Objective metadata
OBJECTIVE_COLS  = ["welfare", "fraction_above_threshold",
                   "welfare_loss_damage", "welfare_loss_abatement"]
MAXIMIZE_COLS   = ["welfare_loss_damage", "welfare_loss_abatement"]
MINIMIZE_COLS   = ["welfare", "fraction_above_threshold"]
OBJECTIVE_LABELS = {
    "welfare":                  "Welfare loss\n(MINIMIZE)",
    "fraction_above_threshold": "Fraction above\n2 C in 2100\n(MINIMIZE)",
    "welfare_loss_damage":      "Welfare loss\nfrom damage\n(MAXIMIZE)",
    "welfare_loss_abatement":   "Welfare loss\nfrom abatement\n(MAXIMIZE)",
}

# DEAP hypervolume helpers
# DEAP requires pure minimisation — MAXIMIZE objectives are negated
_NEGATE = {col: (col in MAXIMIZE_COLS) for col in OBJECTIVE_COLS}

def _deap_hypervolume(archive_df, ref_point):
    Compute hypervolume using DEAP's WFG algorithm (pure minimisation).
    MAXIMIZE objectives are negated before computation.
    ref_point must strictly dominate all converted points.
    
    pts = archive_df[OBJECTIVE_COLS].copy()
    for col, neg in _NEGATE.items():
        if neg:
            pts[col] = -pts[col]
    return deap_hv.hypervolume(pts.values.tolist(), ref_point)

def _compute_ref_point(ref_df, margin=0.1):
    Build a reference point strictly dominated by all solutions.
    For each axis (after minimisation conversion): max + margin * range.
    
    pts = ref_df[OBJECTIVE_COLS].copy()
    for col, neg in _NEGATE.items():
        if neg:
            pts[col] = -pts[col]
    rp = []
    for col in OBJECTIVE_COLS:
        hi = pts[col].max()
        lo = pts[col].min()
        rp.append(hi + margin * max(hi - lo, 1e-6))
    return rp

print(f"JUSTICE root : {_JUSTICE_ROOT}")
print(f"Results root : {RESULTS_ROOT}")
print("DEAP hypervolume + EMA Workbench GD/EI metrics : OK") """

SyntaxError: invalid decimal literal (2387283710.py, line 55)

---

## Step 1 — Load Results from Assignment 5

Scan `results/` for Pareto-front CSVs, convergence archives, and convergence CSVs.  
Directories are named `<welfare_function>_<nfe>_<seed>/`, so we can recover the NFE directly
from the directory name and group all data by it.

Each seed directory contains:
- `pareto_front_<seed>.csv` — final Pareto-optimal solutions (levers + objectives)
- `<welfare_function>_<nfe>_<seed>.tar.gz` — archive snapshots (ArchiveLogger output)
- `convergence_<seed>.csv` — EpsilonProgress & operator probabilities (EMA Workbench)


In [33]:
import os
import glob
import pandas as pd

# We definiëren hard de filters die jij wilt hanteren
TARGET_NFE = 5000
TARGET_SEEDS = [1, 2, 3, 4, 5]

# Dictionaries waarin we alléén de gefilterde data opslaan
all_pareto_fronts = {}  # Structuur: {(nfe, seed): dataframe}
all_snapshots = {}      # Structuur: {(nfe, seed): {checkpoint_nfe: dataframe}}
all_convergence = {}    # Structuur: {(nfe, seed): dataframe}

print(f"Scannen van {RESULTS_ROOT} naar uitsluitend {TARGET_NFE} NFE (Seeds {TARGET_SEEDS})...")

# 1. Pareto fronts inladen
for seed in TARGET_SEEDS:
    # Zoek specifiek naar het bestand voor dit budget en deze seed
    # Dit vangt zowel platte mappen op als submappen
    pattern = os.path.join(RESULTS_ROOT, f"**/*_front_{seed}.csv")
    files = glob.glob(pattern, recursive=True)
    
    # Als dat niks oplevert, zoeken we op de naamgeving binnen submappen van 5000 NFE
    if not files:
        pattern = os.path.join(RESULTS_ROOT, f"*{TARGET_NFE}*_{seed}*", f"pareto_front_{seed}.csv")
        files = glob.glob(pattern, recursive=True)
        
    for f in files:
        df = pd.read_csv(f)
        all_pareto_fronts[(TARGET_NFE, seed)] = df
        print(f"  [GELADEN] Pareto Front: Seed {seed}")

# 2. Archives (.tar.gz snapshots) inladen
for seed in TARGET_SEEDS:
    pattern = os.path.join(RESULTS_ROOT, f"**/*{TARGET_NFE}*_{seed}*.tar.gz")
    files = glob.glob(pattern, recursive=True)
    
    for f in files:
        try:
            # We gebruiken de handige load_archives functie van de docenten uit Cel 1
            snapshots_list = load_archives(f)
            # Zet de lijst om naar een dictionary {checkpoint_nfe: dataframe}
            snapshots_dict = {nfe: df for nfe, df in snapshots_list}
            all_snapshots[(TARGET_NFE, seed)] = snapshots_dict
            print(f"  [GELADEN] Archive Snapshots: Seed {seed} ({len(snapshots_dict)} checkpoints)")
        except Exception as e:
            print(f"  [WAARSCHUWING] Kon archive voor Seed {seed} niet laden: {e}")

# 3. Convergence CSVs inladen (voor epsilon progressie)
for seed in TARGET_SEEDS:
    pattern = os.path.join(RESULTS_ROOT, f"**/*{TARGET_NFE}*_{seed}*", f"convergence_{seed}.csv")
    files = glob.glob(pattern, recursive=True)
    
    if not files:
        # Alternatief patroon als ze plat in de resultatenmap staan
        pattern = os.path.join(RESULTS_ROOT, f"convergence_{seed}.csv")
        files = glob.glob(pattern, recursive=True)

    for f in files:
        df = pd.read_csv(f)
        all_convergence[(TARGET_NFE, seed)] = df
        print(f"  [GELADEN] Convergence CSV: Seed {seed}")

print("\n" + "="*50)
print(f"FILTERSALES SUCCESVOL AFGEROND:")
print(f"-> Aantal Pareto fronts geladen: {len(all_pareto_fronts)}")
print(f"-> Aantal Archives geladen: {len(all_snapshots)}")
print(f"-> Aantal Convergence bestanden: {len(all_convergence)}")
print("="*50)

Scannen van /Users/finnmaingay/epa141a/assignments_ema/results naar uitsluitend 5000 NFE (Seeds [1, 2, 3, 4, 5])...
  [GELADEN] Pareto Front: Seed 1
  [GELADEN] Pareto Front: Seed 1
  [GELADEN] Pareto Front: Seed 1
  [GELADEN] Pareto Front: Seed 1
  [GELADEN] Pareto Front: Seed 1
  [GELADEN] Pareto Front: Seed 2
  [GELADEN] Pareto Front: Seed 2
  [GELADEN] Pareto Front: Seed 2
  [GELADEN] Pareto Front: Seed 2
  [GELADEN] Pareto Front: Seed 3
  [GELADEN] Pareto Front: Seed 3
  [GELADEN] Pareto Front: Seed 3
  [GELADEN] Pareto Front: Seed 3
  [GELADEN] Pareto Front: Seed 4
  [GELADEN] Pareto Front: Seed 4
  [GELADEN] Pareto Front: Seed 4
  [GELADEN] Pareto Front: Seed 4
  [GELADEN] Pareto Front: Seed 5
  [GELADEN] Pareto Front: Seed 5
  [GELADEN] Pareto Front: Seed 5
  [GELADEN] Pareto Front: Seed 5
  [GELADEN] Archive Snapshots: Seed 1 (5 checkpoints)
  [GELADEN] Archive Snapshots: Seed 1 (50 checkpoints)
  [GELADEN] Archive Snapshots: Seed 2 (21 checkpoints)
  [GELADEN] Archive Snapsho

In [ ]:
"""# ── Discover Pareto-front CSVs — grouped by NFE ───────────────────────────────
csv_paths = sorted(glob.glob(
    os.path.join(RESULTS_ROOT, "**", "pareto_front_*.csv"), recursive=True
))

if not csv_paths:
    raise FileNotFoundError(
        f"No pareto_front_*.csv found in {RESULTS_ROOT}.\n"
        "Run Assignment 5 (run_optimization_local.py) first."
    )

# nfe_groups[nfe][seed] = DataFrame
nfe_groups = {}
for path in csv_paths:
    dir_name = os.path.basename(os.path.dirname(path))
    parts    = dir_name.split("_")
    try:
        nfe  = int(parts[-2])
        seed = int(parts[-1])
    except (ValueError, IndexError):
        seed = int(os.path.basename(path).replace("pareto_front_", "").replace(".csv", ""))
        nfe  = 0
    df = pd.read_csv(path)
    df = df[df["welfare"] < 1e5].reset_index(drop=True)
    nfe_groups.setdefault(nfe, {})[seed] = df

# ── Discover convergence archives — keyed by (nfe, seed) ─────────────────────
archive_paths = sorted(glob.glob(
    os.path.join(RESULTS_ROOT, "**", "UTILITARIAN_*.tar.gz"), recursive=True
))

nfe_seed_archives = {}   # (nfe, seed) → path
for p in archive_paths:
    parts    = os.path.basename(p).replace(".tar.gz", "").split("_")
    seed_val = int(parts[-1])
    nfe_val  = int(parts[-2])
    nfe_seed_archives[(nfe_val, seed_val)] = p

# ── Discover convergence CSVs — keyed by (nfe, seed) ─────────────────────────
# Saved by run_optimization_local.py as results/<welfare_function>_<nfe>_<seed>/convergence_<seed>.csv
conv_csv_paths = sorted(glob.glob(
    os.path.join(RESULTS_ROOT, "**", "convergence_*.csv"), recursive=True
))

nfe_seed_convergence = {}   # (nfe, seed) → DataFrame
for p in conv_csv_paths:
    dir_name = os.path.basename(os.path.dirname(p))
    parts    = dir_name.split("_")
    try:
        nfe_val  = int(parts[-2])
        seed_val = int(parts[-1])
    except (ValueError, IndexError):
        seed_val = int(os.path.basename(p).replace("convergence_", "").replace(".csv", ""))
        nfe_val  = 0
    df_conv = pd.read_csv(p, index_col=0)
    nfe_seed_convergence[(nfe_val, seed_val)] = df_conv

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"NFE groups found: {sorted(nfe_groups)}")
print()
rows = []
for nfe in sorted(nfe_groups):
    for seed, df in sorted(nfe_groups[nfe].items()):
        rows.append({
            "nfe_budget":      nfe,
            "seed":            seed,
            "n_solutions":     len(df),
            "has_archive":     (nfe, seed) in nfe_seed_archives,
            "has_conv_csv":    (nfe, seed) in nfe_seed_convergence,
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(f"\n{len(nfe_seed_archives)} archive(s) | {len(nfe_seed_convergence)} convergence CSV(s)") """


## What is a reference set?

A reference set is the best collection of non-dominated solutions found across all your optimisation runs — all seeds, all NFEs combined into one Pareto front. It represents the most complete picture of the trade-off frontier your algorithm was able to discover.

The reference set serves two distinct purposes in this assignment:

1. As a benchmark for convergence metrics
Generational Distance (GD) and Epsilon Indicator (EI) measure how close each seed's archive is to the reference set at each point in the optimisation. Without a reference set you have no target to measure distance from.

2. As input for Assignments 7 and 8
The grand reference set (pooled across all seeds and NFE budgets) is saved to CSV and loaded directly in the visualisation and robustness assignments. It is the agreed-upon "best known" Pareto front for your welfare function that the rest of the analysis is built on.


If you tested different  NFE budgets (e.g. 500 NFE, 20 000 NFE) each gets its own reference set.

Each reference set is saved as `reference_set_<welfare_function>_<nfe>.csv`, if you are first comparing results across different nfes. For computing the grand reference set, simply name it `reference_set_<welfare_function>.csv`

We use `ema_workbench.em_framework.optimization.epsilon_nondominated`, which internally
calls Platypus's `EpsilonBoxArchive`.  Docs: https://emaworkbench.readthedocs.io/en/latest/ema_documentation/em_framework/optimization.html#ema_workbench.em_framework.optimization.epsilon_nondominated


In [ ]:
import os
import glob
import pandas as pd
from ema_workbench.em_framework.optimization import epsilon_nondominated, Problem
from ema_workbench import ScalarOutcome, RealParameter

# ── STAP 1: GERICHT INLADEN VAN ALLÉÉN 5000 NFE EN SEEDS 1 t/m 5 ──────────────
TARGET_NFE = 20000
TARGET_SEEDS = [1, 2, 3, 4, 5]

list_df = []

print(f"Scannen van {RESULTS_ROOT} naar uitsluitend {TARGET_NFE} NFE (Seeds {TARGET_SEEDS})...")

for seed in TARGET_SEEDS:
    pattern = os.path.join(RESULTS_ROOT, f"*{TARGET_NFE}*_{seed}", f"pareto_front_{seed}.csv")
    files = glob.glob(pattern)
    
    if not files:
        pattern = os.path.join(RESULTS_ROOT, f"pareto_front_{TARGET_NFE}_{seed}.csv")
        files = glob.glob(pattern)
        
    for f in files:
        print(f"  [GELADEN] Pareto Front: {os.path.basename(os.path.dirname(f))}/{os.path.basename(f)}")
        df = pd.read_csv(f)
        
        # ── CRUCIALE FIX: Vervang spaties in de kolomnamen door underscores ──
        # Dit voorkomt dat pandas.itertuples() crasht op namen als 'center 0'
        df.columns = [col.replace(" ", "_") for col in df.columns]
        
        list_df.append(df)

if not list_df:
    raise FileNotFoundError(f"Geen Pareto-CSV's gevonden voor {TARGET_NFE} NFE en de juiste seeds!")

# Update ook direct onze metadata kolommen zodat ze matchen met de underscores
CLEAN_OBJECTIVE_COLS = [c.replace(" ", "_") for c in OBJECTIVE_COLS]
CLEAN_MAXIMIZE_COLS  = [c.replace(" ", "_") for c in MAXIMIZE_COLS]

# ── STAP 2: MODELPROBLEEM DYNAMISCH EN SPATIE-VRIJ OPBOUWEN ──────────────────
sample_df = list_df[0]
LEVER_COLS = [col for col in sample_df.columns if col not in CLEAN_OBJECTIVE_COLS]

print(f"\nDynamische kolomdetectie (voorzien van underscores):")
print(f"  - Aantal Levers: {len(LEVER_COLS)} (Voorbeeld: '{LEVER_COLS[0]}')")
print(f"  - Aantal Objectives: {len(CLEAN_OBJECTIVE_COLS)}")

# Bouw de formele lijsten voor het Problem-object zonder spaties
decision_variables = [RealParameter(col, 0, 1) for col in LEVER_COLS]
outcomes = [ScalarOutcome(col, kind=ScalarOutcome.MAXIMIZE if col in CLEAN_MAXIMIZE_COLS else ScalarOutcome.MINIMIZE) for col in CLEAN_OBJECTIVE_COLS]

optimization_problem = Problem("justice_ref", decision_variables=decision_variables, objectives=outcomes)

# De toepasselijke epsilons voor de JUSTICE-doelstellingen
EPSILONS = [50.0, 0.05, 10.0, 10.0] 

# ── STAP 3: EPSILON NON-DOMINATION FILTERING TOEPASSEN ──────────────────
print("\nToepassen van epsilon-nondominated sorting via de EMA Workbench...")

reference_set_df = epsilon_nondominated(
    results=list_df, 
    epsilons=EPSILONS, 
    problem=optimization_problem
)

# Zet de kolomnamen voor het uiteindelijke resultaat weer netjes terug naar spaties
# Mocht dat voor je latere opdrachten (7 en 8) vereist zijn vanuit de docentenscripts!
reference_set_df.columns = [col.replace("_", " ") if col not in CLEAN_OBJECTIVE_COLS else col.replace("_", " ") for col in reference_set_df.columns]

print(f"-> Filteren voltooid! Unieke oplossingen in de Reference Set: {len(reference_set_df)}")

# ── STAP 4: OPSLAAN ALS GRAND REFERENCE SET ──────────────────────────────────
WELFARE_FUNCTION = "Utilitarian"
output_filename = f"reference_set_{WELFARE_FUNCTION.lower()}.csv"
output_path = os.path.join(RESULTS_ROOT, output_filename)

reference_set_df.to_csv(output_path, index=False)

print("=" * 60)
print(f"[SUCCES] De Grand Reference Set is gegenereerd en opgeslagen!")
print(f"Pad: {output_path}")
print("=" * 60)

Scannen van /Users/finnmaingay/epa141a/assignments_ema/results naar uitsluitend 5000 NFE (Seeds [1, 2, 3, 4, 5])...
  [GELADEN] Pareto Front: UTILITARIAN_5000_1/pareto_front_1.csv
  [GELADEN] Pareto Front: UTILITARIAN_50000_1/pareto_front_1.csv
  [GELADEN] Pareto Front: UTILITARIAN_5000_2/pareto_front_2.csv
  [GELADEN] Pareto Front: UTILITARIAN_5000_3/pareto_front_3.csv
  [GELADEN] Pareto Front: UTILITARIAN_5000_4/pareto_front_4.csv
  [GELADEN] Pareto Front: UTILITARIAN_5000_5/pareto_front_5.csv

Dynamische kolomdetectie (voorzien van underscores):
  - Aantal Levers: 244 (Voorbeeld: 'center_0')
  - Aantal Objectives: 4

Toepassen van epsilon-nondominated sorting via de EMA Workbench...
-> Filteren voltooid! Unieke oplossingen in de Reference Set: 1
[SUCCES] De Grand Reference Set is gegenereerd en opgeslagen!
Pad: /Users/finnmaingay/epa141a/assignments_ema/results/reference_set_utilitarian.csv


---

## Step 3 — MOEA Performance Metrics - pre-processing

We compute four convergence / quality metrics from the `load_archives` snapshots, one per NFE group:

| Metric | What it measures | Tool | Direction |
|--------|-----------------|------|-----------|
| **Hypervolume (HV)** | Volume of objective space dominated by the archive | `HypervolumeMetric` | higher = better |
| **Epsilon-progress** | Change in archive size per generation | EMA Workbench convergence CSV | positive = still improving |
| **Generational Distance (GD)** | Mean distance from archive solutions to the group reference set | `GenerationalDistanceMetric` | lower = better |
| **Epsilon Indicator (EI)** | Additive epsilon needed so result epsilon-dominates the group reference set | `EpsilonIndicatorMetric` | lower = better; 0 = full dominance |


**Task 3.1** -- After running all three cells below, answer the reflection questions at the bottom of this notebook.

**Task 3.2** -- For each metric, identify the approximate NFE at which convergence occurs (if it does). Summarise in a table.

In [37]:
# This cell loads the convergence archives produced by run_optimization_local.py.
# Each archive contains snapshots of the Pareto front taken at intervals during the run —
# this is what allows you to track whether the MOEA had converged by the end.
#
# load_archives() reads the .tar.gz archive
# and returns a list of (checkpoint_nfe, DataFrame) pairs, one per snapshot.
# We convert this to a dict keyed by checkpoint NFE for easy lookup.
#
# Run this cell as-is. The output tells you how many snapshots were saved per run
# and how many solutions were in the final archive
# Load convergence archives -- keyed by (nfe, seed)
if not nfe_seed_archives:
    print("No convergence archives found -- skipping metric computation.")
else:
    # all_snapshots[(nfe, seed)] = {checkpoint_nfe: DataFrame}
    all_snapshots = {}

    for (nfe, seed), path in sorted(nfe_seed_archives.items()):
        print(f"  Loading: NFE={nfe:,}  seed={seed} ...", end=" ")
        # load_archives returns list[tuple[int, DataFrame]] in ema_workbench 3.0
        snaps = dict(load_archives(path))
        snaps = {
            n: df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")],
                       errors="ignore")
            for n, df in snaps.items()
        }
        all_snapshots[(nfe, seed)] = snaps
        max_n = max(snaps)
        print(f"{len(snaps)} snapshots | final archive = {len(snaps[max_n])} solutions @ NFE {max_n:,}")

    print(f"\n{len(all_snapshots)} archive(s) loaded.")

  Loading: NFE=5  seed=1 ... 1 snapshots | final archive = 1 solutions @ NFE 100
  Loading: NFE=5  seed=2 ... 1 snapshots | final archive = 1 solutions @ NFE 100
  Loading: NFE=5  seed=3 ... 1 snapshots | final archive = 1 solutions @ NFE 100
  Loading: NFE=5  seed=4 ... 1 snapshots | final archive = 1 solutions @ NFE 100
  Loading: NFE=5  seed=5 ... 1 snapshots | final archive = 1 solutions @ NFE 100
  Loading: NFE=20  seed=1 ... 1 snapshots | final archive = 1 solutions @ NFE 100
  Loading: NFE=20  seed=2 ... 1 snapshots | final archive = 1 solutions @ NFE 100
  Loading: NFE=20  seed=3 ... 1 snapshots | final archive = 1 solutions @ NFE 100
  Loading: NFE=20  seed=4 ... 1 snapshots | final archive = 1 solutions @ NFE 100
  Loading: NFE=20  seed=5 ... 1 snapshots | final archive = 1 solutions @ NFE 100
  Loading: NFE=2,000  seed=2 ... 2 snapshots | final archive = 1 solutions @ NFE 2,008
  Loading: NFE=2,000  seed=3 ... 2 snapshots | final archive = 1 solutions @ NFE 2,012
  Loading: 

## Step 4 -  Computing and visualizing convergence metrics

Compute and visualize MOEA convergence metrics, check the following for guidance: 

https://emaworkbench.readthedocs.io/en/latest/examples/optimization_convergence_analysis.html#Convergence-metrics


In [38]:
# --- YOUR TASK ---
# For each seed and NFE budget, compute Pareto front quality metrics
# at every archive snapshot and store the results 
#
# The snapshots are in all_snapshots — each entry contains the Pareto archive
# as it looked at a specific point during the optimisation run.
# By computing metrics at each snapshot you can track whether the algorithm
# converged before the NFE budget was exhausted.
#
# Sample metrics to compute:
#   - Hypervolume (HV)           — how much of the objective space is dominated
#   - Generational Distance (GD) — how close the archive is to the reference set
#   - Epsilon Indicator (EI)     — how much the reference set still dominates the archive
#   - Epsilon-progress (eps)     — how many new solutions were added at each checkpoint
#
#Store your results
#
# Hint: check the ema_workbench convergence metrics documentation and the
# _deap_hypervolume() function defined in the setup cell. 


# YOUR CODE HERE — compute metrics

# ── Visualisation ─────────────────────────────────────────────────────────────
# Create one figure per NFE budget with four panels, one per metric (HV, GD, EI, eps).
# Plot NFE on the x-axis and the metric value on the y-axis.
# Show all seeds on the same panel
#
# YOUR CODE HERE — plot

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ema_workbench.em_framework.optimization import (
    HypervolumeMetric,
    GenerationalDistanceMetric,
    EpsilonIndicatorMetric
)

# ── 1. VOORBEREIDING: REFERENCE SET & BOUNDING BOX ─────────────────────────
# Zorg dat de grand reference set kolommen correct matchen met je model
ref_objectives = reference_set_df[OBJECTIVE_COLS].to_numpy()

# Bepaal het absolute slechtst mogelijke punt (Nadir point) op basis van de data
# We voegen een kleine marge (+10%) toe zodat er geen data op de grens valt
max_bounds = reference_set_df[OBJECTIVE_COLS].max().to_numpy()
min_bounds = reference_set_df[OBJECTIVE_COLS].min().to_numpy()

# Omdat welfare wordt gemaximaliseerd is 'slecht' daar een lage waarde. 
# Voor de overige geminimaliseerde kosten/schade is 'slecht' een hoge waarde.
reference_point = []
for col in OBJECTIVE_COLS:
    if col == "welfare":
        # Slechtste welvaart = minimum waarde minus 10% marge
        reference_point.append(reference_set_df[col].min() * 0.9)
    else:
        # Slechtste kosten/schade = maximum waarde plus 10% marge
        reference_point.append(reference_set_df[col].max() * 1.1)

reference_point = np.array(reference_point)
print(f"Berekend Reference Point voor Hypervolume Bounding Box:\n  {reference_point}\n")

# Initialiseer de officiële EMA Workbench metrics-objecten
gd_metric = GenerationalDistanceMetric(reference_set_df, model.problem)
ei_metric = EpsilonIndicatorMetric(reference_set_df, model.problem)

# We maken een lijst om alle berekende metrics in te verzamelen
metrics_records = []

# ── 2. METRICS BEREKENEN PER SNAPSHOT ──────────────────────────────────────
print("Starten met het berekenen van de kwaliteitsmetrics per snapshot...")

for (nfe_budget, seed), snapshots in all_snapshots.items():
    print(f"  Verwerken: Budget={nfe_budget}, Seed={seed} ({len(snapshots)} snapshots)")
    
    for checkpoint_nfe, snapshot_df in sorted(snapshots.items()):
        # Extraheer alleen de objectieve kolommen van deze specifieke foto
        snapshot_objectives = snapshot_df[OBJECTIVE_COLS].to_numpy()
        
        # A. Hypervolume berekenen middels de stabiele DEAP-hulpfunctie uit de hint
        try:
            hv_val = _deap_hypervolume(snapshot_df, reference_point, model.problem)
        except NameError:
            # Mocht _deap_hypervolume niet in de setup-cel staan, gebruik de fallback:
            from ema_workbench.em_framework.optimization import HypervolumeMetric
            hv_metric_obj = HypervolumeMetric(reference_set_df, model.problem)
            hv_val = hv_metric_obj.calculate(snapshot_df)
            
        # B. Generational Distance (GD) berekenen
        gd_val = gd_metric.calculate(snapshot_df)
        
        # C. Epsilon Indicator (EI) berekenen
        ei_val = ei_metric.calculate(snapshot_df)
        
        # D. Epsilon Progress (Eps) vissen we direct uit de snapshot-grootte
        # (Hoeveel unieke Pareto-oplossingen heeft het archief op dit punt?)
        eps_progress = len(snapshot_df)
        
        # Sla alle resultaten op in ons overzicht
        metrics_records.append({
            "NFE_Budget": nfe_budget,
            "Seed": seed,
            "Checkpoint_NFE": checkpoint_nfe,
            "Hypervolume": hv_val,
            "Generational_Distance": gd_val,
            "Epsilon_Indicator": ei_val,
            "Epsilon_Progress": eps_progress
        })

# Zet de resultaten om naar een handige DataFrame voor de visualisatie
metrics_df = pd.DataFrame(metrics_records)
print("\n[SUCCES] Alle metrics zijn succesvol berekend en opgeslagen in 'metrics_df'!")

# ── 3. VISUALISATIE (4-PANEL CONVERGENTIE GRAFIEK) ──────────────────────────
# We groeperen de plots per NFE budget (als je er meerdere hebt getest)
unique_budgets = metrics_df["NFE_Budget"].unique()

for budget in unique_budgets:
    budget_data = metrics_df[metrics_df["NFE_Budget"] == budget]
    
    # Maak een grid van 2x2 panelen
    fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
    fig.suptitle(f"MOEA Convergence & Quality Metrics (NFE Budget: {budget:,})", fontsize=16, fontweight='bold')
    
    # Definieer de vier subplots en hun instellingen
    plot_configs = [
        {"col": "Hypervolume",           "ax": axes[0, 0], "title": "Hypervolume (Higher = Better)", "color": "green"},
        {"col": "Generational_Distance", "ax": axes[0, 1], "title": "Generational Distance (Lower = Better)", "color": "darkred"},
        {"col": "Epsilon_Indicator",     "ax": axes[1, 0], "title": "Epsilon Indicator (Lower = Better)", "color": "orange"},
        {"col": "Epsilon_Progress",      "ax": axes[1, 1], "title": "Archive Size / Epsilon Progress (Plateau = Converged)", "color": "blue"}
    ]
    
    # Teken elke seed als een aparte lijn in de panelen
    for seed in budget_data["Seed"].unique():
        seed_data = budget_data[budget_data["Seed"] == seed].sort_values("Checkpoint_NFE")
        
        for config in plot_configs:
            config["ax"].plot(
                seed_data["Checkpoint_NFE"], 
                seed_data[config["col"]], 
                label=f"Seed {seed}", 
                alpha=0.8,
                linewidth=1.5
            )
            
    # Grafiekafwerking (titels, assen, legenda's)
    for config in plot_configs:
        config["ax"].set_title(config["config" if "config" in config else "title"], fontweight='bold')
        config["ax"].grid(True, linestyle="--", alpha=0.5)
        config["ax"].set_ylabel("Metric Value")
        if config["ax"] in axes[1]:
            config["ax"].set_xlabel("Number of Function Evaluations (NFE)")
            
    # Voeg één centrale legenda toe aan de rechterkant
    axes[0, 0].legend(loc="upper left", bbox_to_anchor=(0.02, 0.95), ncol=2)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

ImportError: cannot import name 'HypervolumeMetric' from 'ema_workbench.em_framework.optimization' (/opt/miniconda3/envs/epa141a/lib/python3.12/site-packages/ema_workbench/em_framework/optimization.py)

In [39]:
# Snel checken hoeveel rijen er in je rauwe data zitten vóórdat we filteren
totaal_rijen = sum([len(df) for df in list_df])
print(f"Totaal aantal rauwe oplossingen in list_df: {totaal_rijen}")

# Laat de eerste 3 rijen zien van de eerste seed om de cijfers te bekijken
print("\nVoorbeeld van de doelen in Seed 1:")
print(list_df[0][CLEAN_OBJECTIVE_COLS].head(3))

# Check de unieke waarden in één van de kolommen
print("\nAantal unieke waarden in 'welfare':", list_df[0]['welfare'].nunique())

Totaal aantal rauwe oplossingen in list_df: 6

Voorbeeld van de doelen in Seed 1:
        welfare  fraction_above_threshold  welfare_loss_damage  \
0  30805.052561                       0.8         34306.795803   

   welfare_loss_abatement  
0            73981.400695  

Aantal unieke waarden in 'welfare': 1


In [40]:
import os
import glob
import pandas as pd

# We zoeken NU naar ALLE pareto_front CSV's in je hele resultatenmap
all_files = glob.glob(os.path.join(RESULTS_ROOT, "**", "pareto_front_*.csv"), recursive=True)

print(f"Schatkist-inspectie: We vonden in totaal {len(all_files)} Pareto-bestanden.\n")

for f in all_files:
    df = pd.read_csv(f)
    mapnaam = os.path.basename(os.path.dirname(f))
    bestandsnaam = os.path.basename(f)
    
    print(f"Bestand: {mapnaam}/{bestandsnaam}")
    print(f"  -> Aantal gevonden oplossingen (rijen): {len(df)}")
    if 'welfare' in df.columns:
        print(f"  -> Aantal unieke smaken (unieke welfare): {df['welfare'].nunique()}")
    else:
        # Check met spatie als de underscores er nog niet inzitten
        print(f"  -> Aantal unieke smaken (unieke welfare): {df['welfare loss damage'].nunique() if 'welfare loss damage' in df.columns else 'onbekend'}")
    print("-" * 50)

Schatkist-inspectie: We vonden in totaal 22 Pareto-bestanden.

Bestand: UTILITARIAN_2000_4/pareto_front_4.csv
  -> Aantal gevonden oplossingen (rijen): 1
  -> Aantal unieke smaken (unieke welfare): 1
--------------------------------------------------
Bestand: UTILITARIAN_2000_3/pareto_front_3.csv
  -> Aantal gevonden oplossingen (rijen): 1
  -> Aantal unieke smaken (unieke welfare): 1
--------------------------------------------------
Bestand: UTILITARIAN_2000_2/pareto_front_2.csv
  -> Aantal gevonden oplossingen (rijen): 1
  -> Aantal unieke smaken (unieke welfare): 1
--------------------------------------------------
Bestand: UTILITARIAN_2000_5/pareto_front_5.csv
  -> Aantal gevonden oplossingen (rijen): 1
  -> Aantal unieke smaken (unieke welfare): 1
--------------------------------------------------
Bestand: UTILITARIAN_20_4/pareto_front_4.csv
  -> Aantal gevonden oplossingen (rijen): 1
  -> Aantal unieke smaken (unieke welfare): 1
--------------------------------------------------

Interpret the plots
>your answer here


---

## Reflection Questions

**1. Hypervolume convergence.** Does the hypervolume plateau before the NFE budget is exhausted, or is it still growing? What change to the optimisation setup would you recommend if HV was still rising at the final NFE?

**2. Epsilon-progress.** At what approximate NFE does epsilon-progress first reach zero (or near-zero)? Is this consistent with what you see in the hypervolume curve? Explain.

**3. Seed consistency.** If multiple seeds produce very different final hypervolumes within the same NFE group, what does that suggest about (a) the landscape of the objective space, and (b) the reliability of any single seed as a representative Pareto front?
